In [ ]:

# Install dependencies

!pip install -q ultralytics pyvips

import os
import cv2
import json
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:
def load_yolo_model(model_path):
    return YOLO(model_path)



# تحميل Moondream

def load_moondream_model(
    model_id="vikhyatk/moondream2",
    revision="2025-01-09"
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[Moondream] Device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        revision=revision,
        trust_remote_code=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        revision=revision,
        trust_remote_code=True,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device).eval()

    return model, tokenizer, device



In [ ]:
# عرض صورة

def show_bgr(img, title="Image", figsize=(8, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()



# توقع صورة وحدة

def predict_single_image(model, image_path, conf=0.25, iou=0.45, show=False, save=False, save_dir=None):
    results = model(image_path, conf=conf, iou=iou)
    plotted = results[0].plot()

    if show:
        plt.figure(figsize=(8, 6))
        plt.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
        plt.title(f"Detection: {Path(image_path).name}")
        plt.axis("off")
        plt.show()

    if save and save_dir:
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, Path(image_path).name)
        cv2.imwrite(save_path, plotted)

    boxes = results[0].boxes
    detections = []

    if boxes is not None:
        for box in boxes:
            cls_id = int(box.cls[0].item())
            conf_score = float(box.conf[0].item())
            xyxy = box.xyxy[0].tolist()

            detections.append({
                "class_id": cls_id,
                "class_name": model.names[cls_id],
                "confidence": conf_score,
                "box": xyxy
            })

    return detections




In [ ]:
# قص أفضل سيارة

def crop_best_detection(image_path, detections, target_classes=None, padding=10):
 
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")

    filtered = detections
    if target_classes:
        filtered = [d for d in detections if d["class_name"].lower() in [c.lower() for c in target_classes]]

    if not filtered:
        return None, None

    best = max(filtered, key=lambda x: x["confidence"])
    x1, y1, x2, y2 = map(int, best["box"])

    h, w = img.shape[:2]
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(w, x2 + padding)
    y2 = min(h, y2 + padding)

    crop = img[y1:y2, x1:x2]
    return crop, best



In [ ]:
# أسئلة Moondream

# QUESTIONS = {
#     "brand": "What is the brand/make of this car? Answer with just the brand name.",
#     "model": "What is the model of this car? Answer with just the model name.",
#     "color": "What is the color of this car? Answer with just the color.",
#     "body_type": "What is the body type of this car? (Sedan, SUV, Coupe, Hatchback, Pickup, Van) Answer with one word.",
#     "year": "What is the approximate year or generation of this car? Answer briefly."
# }

QUESTIONS = {
    "brand": "Identify the vehicle make/brand. Provide only the brand name (e.g., Toyota).",
    "model": "Identify the specific model name. Provide only the model name (e.g., Yaris).",
    "color": "Identify the exterior color of the vehicle. Provide only the color name.",
    "body_type": "Classify the vehicle body type. Choose one: (Sedan, SUV, Coupe, Hatchback, Pickup, Van).",
}



def extract_specs_with_moondream(md_model, md_tokenizer, crop_bgr):
    pil_img = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))

    image_embeds = md_model.encode_image(pil_img)

    specs = {}
    for field, question in QUESTIONS.items():
        answer = md_model.answer_question(
            image_embeds=image_embeds,
            question=question,
            tokenizer=md_tokenizer
        )
        specs[field] = answer.strip()

    return specs



# تحليل صورة واحدة: YOLO ثم Moondream
 
def analyze_single_image_with_moondream(
    yolo_model,
    md_model,
    md_tokenizer,
    image_path,
    conf=0.25,
    iou=0.45,
    show_detection=True,
    show_crop=True,
    save_crop=False,
    crop_dir="crops",
    target_classes=None
):
    detections = predict_single_image(
        model=yolo_model,
        image_path=image_path,
        conf=conf,
        iou=iou,
        show=show_detection,
        save=False
    )

    if len(detections) == 0:
        print("No detections found.")
        return None

    crop, best_det = crop_best_detection(
        image_path=image_path,
        detections=detections,
        target_classes=target_classes,
        padding=10
    )

    if crop is None:
        print("No target detection found after filtering.")
        return None

    if show_crop:
        show_bgr(crop, title=f"Best Crop: {best_det['class_name']} | conf={best_det['confidence']:.2f}")

    if save_crop:
        os.makedirs(crop_dir, exist_ok=True)
        crop_path = os.path.join(crop_dir, f"{Path(image_path).stem}_crop.jpg")
        cv2.imwrite(crop_path, crop)
        print(f"Crop saved to: {crop_path}")

    specs = extract_specs_with_moondream(md_model, md_tokenizer, crop)

    print("\nVehicle Specs")
    print(f"Color      : {specs['color']}")
    print(f"Body Type  : {specs['body_type']}")
    print(f"Make       : {specs['brand']}")
    print(f"Model      : {specs['model']}")


    return {
        "image_name": Path(image_path).name,
        "best_detection": best_det,
        "specs": specs
    }


In [ ]:
def test_folder_with_moondream(
    yolo_model,
    md_model,
    md_tokenizer,
    folder_path,
    conf=0.25,
    iou=0.45,
    show_every=1,
    target_classes=None
):
    folder = Path(folder_path)
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_paths = [p for p in folder.iterdir() if p.suffix.lower() in image_exts]

    all_results = []

    for i, image_path in enumerate(image_paths):
        print("\n" + "=" * 60)
        print(f"Image: {image_path.name}")

        show_now = (i % show_every == 0)

        result = analyze_single_image_with_moondream(
            yolo_model=yolo_model,
            md_model=md_model,
            md_tokenizer=md_tokenizer,
            image_path=str(image_path),
            conf=conf,
            iou=iou,
            show_detection=show_now,
            show_crop=show_now,
            save_crop=True,
            crop_dir="crops",
            target_classes=target_classes
        )

        all_results.append(result)

    return all_results



In [ ]:
# MAIN
if __name__ == "__main__":
    model_path = "/content/best-6.pt" 
    test_images_folder = "/content/test_images"

    #  YOLO
    yolo_model = load_yolo_model(model_path)

    #  Moondream
    md_model, md_tokenizer, device = load_moondream_model()

    target_classes = None

    results_summary = test_folder_with_moondream(
        yolo_model=yolo_model,
        md_model=md_model,
        md_tokenizer=md_tokenizer,
        folder_path=test_images_folder,
        conf=0.4,
        iou=0.5,
        show_every=1,
        target_classes=target_classes
    )

    print("\n" + "=" * 60)
    print("FINAL JSON SUMMARY")
    print("=" * 60)
    print(json.dumps(results_summary, ensure_ascii=False, indent=2))